# 01 · Define & Explore — protein–NA recognition, LigandMPNN-NA theory, target motif

**Standard slot:** *define & explore.* **For Project 23 this means:** choose a **DNA/RNA target
motif**, understand how proteins recognize nucleic acids (and why **LigandMPNN** — which conditions
on nucleic-acid atoms — is the right sequence designer), write down the protein–NA metrics + cutoffs,
and run a deterministic **mock** mini-run as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real campaign (RFdiffusion near the NA + protein–NA
complex modeling) wants an **A100** (see `MANUAL.md §2`); everything here runs on a no-GPU **mock**
backend so you can build the plumbing anywhere, then switch to the real backend on Colab Pro / A100.

> **The one hard truth of this project:** a protein can stick to *any* DNA/RNA backbone (the
> phosphates are negative and generic) without reading the intended bases. **Sequence specificity —
> preferring your motif over a scrambled one — is the real challenge**, and it is harder than
> protein–protein binding. Every result here is reported against a **scrambled-motif** control.

## Protein–nucleic-acid recognition in one screen

Proteins read DNA/RNA through a mix of: **base-specific** contacts (H-bonds / van der Waals from side
chains to the edges of bases, mostly in the **major groove** of DNA), **shape readout** (the
sequence-dependent width/curvature of the groove), and **backbone** contacts to the negatively-charged
phosphates (strong, but **generic** — they do *not* encode specificity). Natural motifs that do this:
helix-turn-helix, zinc fingers, leucine zippers (DNA); RRM, dsRBD, PUF repeats (RNA).

**Why LigandMPNN, not ProteinMPNN.** ProteinMPNN designs a sequence from protein backbone context
only — it is **blind to the nucleic acid**, so it cannot place residues to read specific bases.
**LigandMPNN conditions on non-protein atoms, including DNA/RNA**, so at the interface it chooses
residues that contact the bases/backbone you actually want recognized. That difference is the whole
point of this project; notebook 04 benchmarks the two head-to-head at the interface.

## The protein–NA metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the protein | thermostability / ΔG |
| **pae_interaction** | Å | complex-model error across the **protein–NA interface** (key complex metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| **specificity_score (dScore)** | REU-like | interface score on **scrambled** motif − on **intended** motif (higher = prefers your motif) | a measured ΔΔG |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

We reuse the shared `"binder"` cutoffs (scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10) for the
**confidence** layers, and add a **specificity gate** (dScore ≥ margin) on top — because for protein–NA
binders, *confidence is not specificity*. `pae_interaction` low does **not** mean it binds your motif;
a passing design is a **hypothesis** until an EMSA / fluorescence-anisotropy assay with a
**scrambled-NA control** (notebook 05).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Choose the DNA/RNA target motif

The design target is a **nucleic-acid motif** plus (ideally) a structure of a known protein–NA complex
to scaffold near. Fetch a *candidate* protein–NA complex with `data/download_data.py` (**verify the
accession on RCSB** — protein–DNA *and* protein–RNA complexes exist; pick one that matches your goal),
and **you pick the DNA/RNA target motif** (a transcription-factor box, an operator, an RNA hairpin
sequence, ...) from the complex or the literature.

Below we just *declare* an EXAMPLE DNA motif so the notebook runs end-to-end; **replace it with the
motif you derive and verify**. DNA uses A/C/G/T; RNA uses A/C/G/U. `normalize_motif()` validates the
alphabet so typos fail loudly (it rejects IUPAC ambiguity codes — expand them to a concrete sequence).

In [ ]:
import na_binder_tools as nbt

NA_TYPE = "DNA"                         # "DNA" or "RNA" — match your target and complex
# EXAMPLE target motif — VERIFY/REPLACE from your chosen complex/literature (data/README.md).
# A CRE-like 8-bp box, strict ACGT, so the plumbing runs; real numbering/sequence depends on your target.
MOTIF = nbt.normalize_motif("TGACGTCA", NA_TYPE)   # EXAMPLE_DATA placeholder motif
SCRAMBLED = nbt.scramble_motif(MOTIF, seed=0)      # specificity control (same bases, shuffled order)

print("na_type :", NA_TYPE)
print("motif   :", MOTIF, " (EXAMPLE — replace with your verified DNA/RNA target motif)")
print("scramble:", SCRAMBLED, " (specificity control — same composition, different order)")

## 2 · Mock hello-world: scaffold → LigandMPNN → model → specificity

`scripts/na_binder_tools.py` exposes the whole NA-binder pipeline behind one API:
`scaffold_near_na(...)` (RFdiffusion near the NA), `ligandmpnn_na(...)` (NA-aware sequence design — the
central step), `model_complex(...)` (protein–NA complex modeling), and `motif_specificity(...)`
(intended vs scrambled motif). The **mock** backend is deterministic and GPU-free so you can develop
the plumbing. **Never report mock numbers as real** — they are `SYNTHETIC` by construction.

In [ ]:
# One backbone -> two NA-aware sequences -> model the complex -> score specificity. All SYNTHETIC.
backbones = nbt.scaffold_near_na(MOTIF, n=2, tool="mock", na_type=NA_TYPE)
designs = nbt.ligandmpnn_na(backbones[0], MOTIF, n=2, tool="mock", na_type=NA_TYPE, seq_tool="ligandmpnn")
nbt.score_designs(designs, na=MOTIF, tool="mock")          # protein-NA complex -> pae_interaction, plddt, scrmsd
nbt.add_specificity(designs, MOTIF, scrambled=SCRAMBLED, tool="mock")   # intended vs scrambled motif

d = designs[0]
print("example LigandMPNN (NA-aware) design:")
print("  id   :", d.design_id)
print("  len  :", d.length, "aa   na_type:", d.na_type, "  motif:", d.target_motif)
print("  seq  :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd, " plddt =", d.plddt, " (SYNTHETIC)")
print("  dG_motif =", d.dG_motif, " dG_scrambled =", d.dG_scrambled,
      " specificity_score =", d.specificity_score, " is_specific =", d.is_specific, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'rfdiffusion'/'ligandmpnn'/'boltz' on Colab (A100). See MANUAL.md §2.")

## 3 · Why the scrambled-motif control matters (specificity proxy)

A binder only **reads your motif** if it scores better on the intended motif than on a scrambled one
(same base composition, different order). `motif_specificity()` returns `dScore = score(scrambled) −
score(intended)`; **positive and large ⇒ prefers your motif ⇒ specific**. A design with great
`pae_interaction` but `dScore ≈ 0` is a **non-specific backbone-gripper** — common, and the honest
hard truth of protein–NA design. This is a *computational proxy*; the wet-lab scrambled-NA
EMSA/anisotropy control (notebook 05) is what actually tests it.

In [ ]:
for b in designs:
    tag = "SPECIFIC" if b.is_specific else "non-specific"
    print(f"{b.design_id}: dG_motif={b.dG_motif}  dG_scram={b.dG_scrambled}  "
          f"dScore={b.specificity_score}  -> {tag} (SYNTHETIC)")
print("\nMany designs will be non-specific — that is expected and must be reported honestly.")

## Visualize a protein–NA complex (py3Dmol)

Use this to eyeball a predicted protein–NA complex once you have a real PDB (from Boltz-2 / AF3-style
modeling): protein cartoon + nucleic-acid sticks, so you can see whether the protein sits in the major
groove / on the bases (specific) or just along the backbone (non-specific).

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})                 # protein
    view.addStyle({"resn": ["DA","DT","DG","DC","A","U","G","C"]},    # nucleic acid
                  {"stick": {}})
    view.zoomTo()
    return view.show()

# Example (after a real Boltz-2 / AF3-style prediction writes a protein-NA complex PDB):
# show_complex("results/model/top_complex.pdb")
print("show_complex(pdb_path) ready (protein cartoon + nucleic-acid sticks).")

## D0 checklist
- [ ] Protein–NA complex accession verified on RCSB (the `data/` candidate is a *candidate* — confirm it is the right protein–DNA/RNA complex, chains, resolution).
- [ ] **DNA/RNA target motif chosen** (from the complex/literature, not invented) and its scrambled control written down.
- [ ] One-paragraph definition of each protein–NA metric **with** its "does not mean" note (esp. confidence ≠ specificity).
- [ ] Reproduced mock mini-run (scaffold → LigandMPNN → model → specificity) with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls (incl. the **scrambled-NA** control); `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — scaffold near the NA + LigandMPNN design around it.